In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")

tf = None
keras = None
L = None
TF_AVAILABLE = False

os.environ.setdefault("OSIC_TRY_TF", "0")

try:
    if os.environ.get("OSIC_TRY_TF", "0") == "1":
        import tensorflow as tf  # noqa: F401

        TF_AVAILABLE = True
        tf.random.set_seed(SEED)
        try:
            tf.config.threading.set_intra_op_parallelism_threads(1)
            tf.config.threading.set_inter_op_parallelism_threads(1)
        except Exception:
            pass

        from tensorflow import keras  # noqa: E402
        from tensorflow.keras import layers as L  # noqa: E402

        print("TensorFlow version:", tf.__version__)
    else:
        raise ImportError("TF disabled for stability; using NumPy fallback.")
except Exception as e:
    TF_AVAILABLE = False
    tf = None
    keras = None
    L = None
    print("TensorFlow unavailable; using NumPy fallback. Root error:", repr(e))



TensorFlow unavailable; using NumPy fallback. Root error: ImportError('TF disabled for stability; using NumPy fallback.')


In [2]:
raw_test = pd.read_csv("/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv")
X_prediction = pd.read_csv(
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)



In [3]:
TEST_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression/test"

DESIRED_SIZE = (30, 256, 256)
BATCH_SIZE = 256



In [4]:
X_prediction["Patient"] = X_prediction["Patient_Week"].str.extract(r"(.*)_.*")
X_prediction["Weeks"] = X_prediction["Patient_Week"].str.extract(r".*_(.*)").astype(int)
X_prediction = X_prediction[["Patient", "Weeks", "Patient_Week"]]

raw_test_base = raw_test.rename(
    columns={
        "Weeks": "Base_week",
        "Percent": "Base_percent",
        "FVC": "Base_FVC",
    }
)

X_prediction = X_prediction.merge(raw_test_base, how="left", on="Patient")[
    [
        "Patient",
        "Base_week",
        "Base_FVC",
        "Base_percent",
        "Age",
        "Sex",
        "SmokingStatus",
        "Weeks",
        "Patient_Week",
    ]
].reset_index(drop=True)



In [5]:
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.exceptions import NotFittedError


class OneHotEncoder(SklearnOneHotEncoder):
    def __init__(self, **kwargs):
        super(OneHotEncoder, self).__init__(**kwargs)
        self.fit_flag = False

    def fit(self, X, **kwargs):
        out = super().fit(X)
        self.fit_flag = True
        return out

    def transform(self, X, categories, index="", name="", **kwargs):
        sparse_matrix = super(OneHotEncoder, self).transform(X)
        new_columns = self.get_new_columns(X=X, name=name, categories=categories)
        d_out = pd.DataFrame(sparse_matrix.toarray(), columns=new_columns, index=index)
        return d_out

    def fit_transform(self, X, categories, index, name, **kwargs):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)

    def get_new_columns(self, X, name, categories):
        new_columns = []
        for j in range(len(categories)):
            new_columns.append("{}_{}".format(name, categories[j]))
        return new_columns


class data_preparation:
    def __init__(self):
        self.enc_sex = LabelEncoder()
        self.enc_smok = LabelEncoder()
        self.onehotenc_smok = OneHotEncoder()

    def __call__(self, data_untransformed):
        data = data_untransformed.copy(deep=True)

        try:
            data["Sex"] = self.enc_sex.transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.transform(
                data["SmokingStatus"].values
            )
            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )
        except NotFittedError:
            data["Sex"] = self.enc_sex.fit_transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.fit_transform(
                data["SmokingStatus"].values
            )
            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.fit_transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )

        return data




In [6]:
data_prep = data_preparation()

raw_train = pd.read_csv("/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv")
_fit_df = pd.concat(
    [
        raw_train[["Sex", "SmokingStatus"]].dropna(),
        raw_test[["Sex", "SmokingStatus"]].dropna(),
    ],
    axis=0,
    ignore_index=True,
)

_dummy = pd.DataFrame(
    {"Sex": _fit_df["Sex"].values, "SmokingStatus": _fit_df["SmokingStatus"].values}
)
data_prep(_dummy)

X_prediction = data_prep(X_prediction).sort_values("Patient").reset_index(drop=True)

for col in ["_Currently smokes", "_Ex-smoker", "_Never smoked"]:
    if col not in X_prediction.columns:
        X_prediction[col] = 0




In [7]:
def sort_function(x):
    return int(x.split(".")[0])


_VOL_CACHE = {}  # (path, patient, desired_size) -> np.ndarray [D,H,W] float32
_PATIENT_FILES_CACHE = {}  # (path, patient) -> list[str] sorted filenames
_RESIZE_FACTORS_CACHE = {}  # (patient, desired_size) -> factors np.ndarray float32


def _get_sorted_patient_files(path, patient):
    key = (path, patient)
    files = _PATIENT_FILES_CACHE.get(key)
    if files is None:
        files = sorted(os.listdir(os.path.join(path, patient)), key=sort_function)
        _PATIENT_FILES_CACHE[key] = files
    return files


def _read(path, patients=[], desired_size=(60, 512, 512)):
    import pydicom
    from scipy.ndimage import zoom

    patients = list(patients)
    n_pat = len(patients)

    desired_size = tuple(int(x) for x in desired_size)

    X = np.empty(
        (n_pat, 1, desired_size[0], desired_size[1], desired_size[2]),
        dtype=np.float32,
    )

    for i, patient in enumerate(patients):
        key = (path, patient, desired_size)
        arr = _VOL_CACHE.get(key)
        if arr is None:
            list_patient_files = _get_sorted_patient_files(path, patient)
            folder = os.path.join(path, patient)

            fp0 = os.path.join(folder, list_patient_files[0])
            ds0 = pydicom.dcmread(fp0, force=True)
            first = ds0.pixel_array.astype(np.float32, copy=False)

            n_slices = len(list_patient_files)
            stack = np.empty((n_slices,) + first.shape, dtype=np.float32)
            stack[0] = first

            for j in range(1, n_slices):
                fp = os.path.join(folder, list_patient_files[j])
                ds = pydicom.dcmread(fp, force=True)
                stack[j] = ds.pixel_array.astype(np.float32, copy=False)

            arr = stack
            arr = (arr - np.mean(arr)) / (np.std(arr) + 1e-6)

            fkey = (patient, desired_size)
            factors = _RESIZE_FACTORS_CACHE.get(fkey)
            if factors is None:
                factors = np.asarray(desired_size, dtype=np.float32) / np.asarray(
                    arr.shape, dtype=np.float32
                )
                _RESIZE_FACTORS_CACHE[fkey] = factors

            arr = zoom(arr, factors, mode="nearest").astype(np.float32, copy=False)
            _VOL_CACHE[key] = arr

        X[i, 0] = arr

    return X




In [8]:
temp_SELECTED_COLUMNS = [
    "Weeks",
    "Base_week",
    "Base_FVC",
    "Base_percent",
    "Age",
    "Sex",
    "_Currently smokes",
    "_Ex-smoker",
    "_Never smoked",
]

from math import ceil

if TF_AVAILABLE:
    _BaseSeq = keras.utils.Sequence
else:
    _BaseSeq = object


class DataGenerator(_BaseSeq):
    def on_epoch_end(self):
        self.indices = np.arange(len(self.list_IDs))

    def __len__(self):
        return int(ceil(len(self.indices) / self.batch_size))

    def __init__(
        self,
        train,
        list_IDs,
        batch_size=1,
        desired_size=(10, 512, 512),
        img_path=TEST_PATH,
        *args,
        **kwargs
    ):
        self.train = train
        self.list_IDs = np.asarray(list_IDs, dtype=np.int64)
        self.batch_size = int(batch_size)
        self.desired_size = desired_size
        self.img_path = img_path

        self._patients = self.train["Patient"].to_numpy()
        self._tab = np.asarray(self.train[temp_SELECTED_COLUMNS], dtype=np.float32)

        self.on_epoch_end()

    def __getitem__(self, index):
        idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        row_ids = self.list_IDs[idx]

        batch_patients = self._patients[row_ids]

        mapping = {}
        patients_in_batch = []
        inv = np.empty(batch_patients.shape[0], dtype=np.int64)
        for k, p in enumerate(batch_patients):
            pos = mapping.get(p)
            if pos is None:
                pos = len(patients_in_batch)
                mapping[p] = pos
                patients_in_batch.append(p)
            inv[k] = pos

        imgs = _read(
            self.img_path, patients=patients_in_batch, desired_size=self.desired_size
        )
        imgs = np.moveaxis(imgs, 1, -1)  # [P,D,H,W,1]

        tab = self._tab[row_ids]  # [B, n_tab]
        return row_ids, batch_patients, tab, imgs, inv

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]




In [9]:
pred_SELECTED_COLUMNS = [
    "Weeks",
    "Base_week",
    "Patient",
    "Base_FVC",
    "Base_percent",
    "Age",
    "Sex",
    "_Currently smokes",
    "_Ex-smoker",
    "_Never smoked",
]

pred_generator = DataGenerator(
    X_prediction[pred_SELECTED_COLUMNS],
    X_prediction.index.tolist(),
    batch_size=BATCH_SIZE,
    desired_size=DESIRED_SIZE,
    img_path=TEST_PATH,
)



In [10]:
MODEL_DIR = "/kaggle/input/cnn-for-latent-features"


def _has_saved_model(p):
    return os.path.isdir(p) and (
        os.path.exists(os.path.join(p, "saved_model.pb"))
        or os.path.exists(os.path.join(p, "saved_model.pbtxt"))
    )


def _tfsmlayer_call(layer, x):
    out = layer(x)
    if isinstance(out, dict):
        out = out[next(iter(out.keys()))]
    return out


cnn_path = os.path.join(MODEL_DIR, "CNN_0")
mlp_path = os.path.join(MODEL_DIR, "model_0")

use_external = (
    TF_AVAILABLE and _has_saved_model(cnn_path) and _has_saved_model(mlp_path)
)

CNN = None
model = None

if use_external:

    def _get_serving_endpoint(model_path):
        imported = tf.saved_model.load(model_path)
        if "serving_default" in imported.signatures:
            return "serving_default"
        return list(imported.signatures.keys())[0]

    cnn_endpoint = _get_serving_endpoint(cnn_path)
    mlp_endpoint = _get_serving_endpoint(mlp_path)
    CNN = keras.layers.TFSMLayer(cnn_path, call_endpoint=cnn_endpoint)
    model = keras.layers.TFSMLayer(mlp_path, call_endpoint=mlp_endpoint)
    print("Loaded external SavedModels.")
else:
    train_df = raw_train.copy()
    train_df = train_df.dropna(subset=["Patient", "Weeks", "FVC"])
    train_df["Weeks"] = train_df["Weeks"].astype(int)

    slopes = {}
    intercepts = {}
    global_slope = -10.0  # mild decline prior; used if patient history insufficient
    global_intercept_shift = 0.0

    wk = train_df["Weeks"].to_numpy(np.float32)
    fvc = train_df["FVC"].to_numpy(np.float32)
    x = wk - wk.mean()
    denom = float((x * x).sum())
    if denom > 0:
        global_slope = float(((x) * (fvc - fvc.mean())).sum() / denom)
        global_intercept_shift = float(fvc.mean() - global_slope * wk.mean())

    for pid, g in train_df.groupby("Patient", sort=False):
        if len(g) < 2:
            continue
        w = g["Weeks"].to_numpy(np.float32)
        y = g["FVC"].to_numpy(np.float32)
        x = w - w.mean()
        denom = float((x * x).sum())
        if denom <= 0:
            continue
        a = float((x * (y - y.mean())).sum() / denom)  # slope
        b = float(y.mean() - a * w.mean())  # intercept
        slopes[pid] = a
        intercepts[pid] = b

    # Change (metric-aligned, minimal): compute a robust Laplace-optimal sigma using per-patient residuals,
    # then take a robust median and lightly shrink toward 70 to reduce over-conservative Confidence that hurts -log(sigma).
    per_patient_sigma = []
    for pid, g in train_df.groupby("Patient", sort=False):
        w = g["Weeks"].to_numpy(np.float32)
        y = g["FVC"].to_numpy(np.float32)
        if pid in slopes:
            a = slopes[pid]
            b = intercepts[pid]
        else:
            a = global_slope
            b = global_intercept_shift
        pred = a * w + b
        r = np.abs(y - pred)
        if r.size >= 2:
            mae_pid = float(np.mean(r))
            sigma_pid = float(np.sqrt(2.0) * mae_pid)
            per_patient_sigma.append(sigma_pid)

    if len(per_patient_sigma) == 0:
        sigma_from_laplace = 200.0
    else:
        sigma_from_laplace = float(
            np.median(np.asarray(per_patient_sigma, dtype=np.float32))
        )

    # Light shrink to reduce overestimation while keeping stability; clipping respects metric sigma>=70 behavior.
    base_sigma = float(np.clip(0.92 * sigma_from_laplace, 70.0, 500.0))

    patient_slope = {}
    for pid in X_prediction["Patient"].unique():
        if pid in slopes:
            patient_slope[pid] = slopes[pid]
        else:
            patient_slope[pid] = global_slope

    def fallback_predict(tab, patients):
        weeks = tab[:, 0]
        base_week = tab[:, 1]
        base_fvc = tab[:, 2]
        out = np.empty((tab.shape[0], 3), dtype=np.float32)
        for i, pid in enumerate(patients):
            a = float(patient_slope.get(pid, global_slope))
            mu = float(base_fvc[i] + a * (weeks[i] - base_week[i]))

            horizon = abs(float(weeks[i] - base_week[i]))
            sigma = base_sigma + 0.12 * horizon
            sigma = float(np.clip(sigma, 70.0, 1000.0))

            out[i, 1] = mu
            out[i, 0] = mu - sigma / 2.0
            out[i, 2] = mu + sigma / 2.0
        return out

    print("Using calibrated linear NumPy fallback (TF/SavedModel not available).")
    print(
        "Fallback base_sigma:",
        base_sigma,
        "(robust Laplace from per-patient residuals; raw median sigma≈",
        sigma_from_laplace,
        ")",
    )



Using calibrated linear NumPy fallback (TF/SavedModel not available).
Fallback base_sigma: 91.03439025878906 (robust Laplace from per-patient residuals; raw median sigma≈ 98.95042419433594 )


In [11]:
y_prediction = np.empty((X_prediction.shape[0], 3), dtype=np.float32)

for row_ids, batch_patients, tab, imgs, inv in pred_generator:
    if use_external:
        out_imgs = _tfsmlayer_call(CNN, tf.convert_to_tensor(imgs, dtype=tf.float32))
        out_imgs_np = out_imgs.numpy()  # [P, latent]
        X_imgs = out_imgs_np[inv]  # [B, latent] aligned to rows
        mlp_inp = np.concatenate([X_imgs, tab], axis=1)
        out = _tfsmlayer_call(
            model, tf.convert_to_tensor(mlp_inp, dtype=tf.float32)
        ).numpy()
    else:
        out = fallback_predict(tab.astype(np.float32, copy=False), batch_patients)

    y_prediction[row_ids] = out

assert np.isfinite(y_prediction).all()



In [12]:
pred_fvc = y_prediction[:, 1].astype(np.float32, copy=False)

pred_sigma = (y_prediction[:, 2] - y_prediction[:, 0]).astype(np.float32, copy=False)
pred_conf = np.clip(pred_sigma, 70.0, 2000.0)

sub = pd.DataFrame(
    data={
        "Patient_Week": X_prediction["Patient_Week"].values,
        "FVC": pred_fvc,
        "Confidence": pred_conf,
    }
)

sample_sub = pd.read_csv(
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)
sub = sample_sub[["Patient_Week"]].merge(sub, on="Patient_Week", how="left")
assert sub.shape[0] == sample_sub.shape[0]
assert list(sub.columns) == ["Patient_Week", "FVC", "Confidence"]

if sub[["FVC", "Confidence"]].isna().any().any():
    raise RuntimeError(
        "Submission contains NaNs after merge; check Patient_Week alignment."
    )

sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)
print("FVC range:", float(np.nanmin(sub["FVC"])), float(np.nanmax(sub["FVC"])))
print(
    "Confidence range:",
    float(np.nanmin(sub["Confidence"])),
    float(np.nanmax(sub["Confidence"])),
)

                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2386.826172   93.554443
1  ID00126637202218610655908_-2  2386.262939   93.434326
2  ID00126637202218610655908_-1  2385.699951   93.314209
3   ID00126637202218610655908_0  2385.136719   93.194336
4   ID00126637202218610655908_1  2384.573486   93.074463
Wrote submission.csv with shape: (1908, 3)
FVC range: 1532.8797607421875 6400.689453125
Confidence range: 91.0341796875 103.63427734375
